In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import os
import matplotlib.pyplot as plt
from collections import OrderedDict
from PIL import Image
import matplotlib.pyplot as plt
import random
import sys


# ======================= Import dataset utilities ========================
os.chdir("/content/drive/MyDrive/Colab Notebooks/NVDIA_Project/Image-level-micro-gesture-classification")
from dataset import get_train_loader, get_train_dataset

# ======================= Import model utilities ========================
from models.resnet18_model import build_resnet18

In [8]:
# Connection check to GPU and check if it's available
print(torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
!nvidia-smi

True
Wed Mar 18 14:43:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

In [ ]:
# This is loading data from Google Drive into colab. I don't recommend it
# train_dir = "/content/drive/MyDrive/Colab Notebooks/NVDIA_Project/Image-level-micro-gesture-classification/data/train"

# Betterway is uploading images as a zip file into colan then unzipping it in this directory. after that you can use this address to load data
train_dir = "/content/train"
trainloader, valloader = get_train_loader(train_dir=train_dir, batch_size=256, val_split=0.25)

In [12]:
# Step 3: Model Selection - Test loading a pre-trained model
model = build_resnet18()
model.to(device)
print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 161MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

In [15]:
print(len(trainloader))

178


In [16]:
num_epochs = 5
train_losses, val_losses, train_acc_list, val_acc_list = [], [], [], []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for idx, (inputs, labels) in enumerate(trainloader):

        if (idx+1) % 20 == 0:
            print(f"Batch {idx+1}/{len(trainloader)}")

        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(trainloader.dataset)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_acc_list.append(train_acc)

    model.eval()
    val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for idx, (inputs, labels) in enumerate(valloader):

            if (idx+1) % 20 == 0:
                print(f"Batch {idx+1}/{len(valloader)}")

            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = val_loss / len(valloader.dataset)
    val_acc = 100. * correct / total
    val_losses.append(val_loss)
    val_acc_list.append(val_acc)

    scheduler.step(val_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}')

Batch 20/178
Batch 40/178
Batch 60/178
Batch 80/178
Batch 100/178
Batch 120/178
Batch 140/178
Batch 160/178
Batch 20/60
Batch 40/60
Batch 60/60
Epoch [1/5] Train Loss: 3.7174 | Train Acc: 25.40% | Val Acc: 27.58


/tmp/ipykernel_2607/34786832.py:52: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(val_loss)


Batch 20/178
Batch 40/178
Batch 60/178
Batch 80/178
Batch 100/178
Batch 120/178
Batch 140/178
Batch 160/178
Batch 20/60
Batch 40/60
Batch 60/60
Epoch [2/5] Train Loss: 2.5593 | Train Acc: 27.20% | Val Acc: 27.60
Batch 20/178
Batch 40/178
Batch 60/178
Batch 80/178
Batch 100/178
Batch 120/178
Batch 140/178
Batch 160/178
Batch 20/60
Batch 40/60
Batch 60/60
Epoch [3/5] Train Loss: 2.4960 | Train Acc: 28.02% | Val Acc: 27.78
Batch 20/178
Batch 40/178
Batch 60/178
Batch 80/178
Batch 100/178
Batch 120/178
Batch 140/178
Batch 160/178
Batch 20/60
Batch 40/60
Batch 60/60
Epoch [4/5] Train Loss: 2.4168 | Train Acc: 29.27% | Val Acc: 28.52
Batch 20/178
Batch 40/178
Batch 60/178
Batch 80/178
Batch 100/178
Batch 120/178
Batch 140/178
Batch 160/178
Batch 20/60
Batch 40/60
Batch 60/60
Epoch [5/5] Train Loss: 2.3150 | Train Acc: 31.60% | Val Acc: 27.31


In [ ]:
save_path = "outputs/checkpoints/resnet18_micro_gesture.pth"

torch.save(model.state_dict(), save_path)
print(f"Model saved to: {save_path}")